# Monte Carlo Policy Gradient (MCPG)

MCPG searches for a best binary vector. This section states that search problem, then describes the ideal answer as a distribution $q^*$.

**Citation:** Chen, C., Chen, R., Li, T., Ao, R., & Wen, Z. (2023). *A Monte Carlo Policy Gradient Method with Local Search for Binary Optimization*. [arXiv:2307.00783](https://arxiv.org/abs/2307.00783).

## Part 1 Problem setup

MCPG solves the binary optimization problem

$$
\min_{x \in \{-1,+1\}^{n}} f(x).
$$

The unknown is a vector $x = (x_1,\ldots,x_n)$ whose entries are restricted to $\{-1,+1\}$. The function $f$ gives the cost of each such vector. A solution is optimal when its cost equals the minimum value $f^{*}$.

### The ideal distribution $q^{*}$

Let $X^{*}$ be the set of all optimal solutions,

$$
X^{*} = \{\, x \in \{-1,+1\}^{n} : f(x) = f^{*} \,\}.
$$

The ideal sampler draws only from that set, and it draws each optimal vector with the same probability:

$$
q^{*}(x) =
\begin{cases}
1/|X^{*}| & \text{if } x \in X^{*}, \\
0 & \text{if } x \notin X^{*}.
\end{cases}
$$

One sample from $q^{*}$ is a minimizer of $f$. The distribution cannot be used directly: constructing it requires $X^{*}$, which is the set the algorithm is trying to find.

### The Gibbs distribution $q_{\lambda}$

The paper approaches $q^{*}$ with the Gibbs distribution at temperature $\lambda > 0$,

$$
q_{\lambda}(x) = \frac{1}{Z_{\lambda}} \exp\!\left(-\frac{f(x)}{\lambda}\right),
\qquad
Z_{\lambda} = \sum_{x \in \{-1,+1\}^{n}} \exp\!\left(-\frac{f(x)}{\lambda}\right).
$$

Vectors with smaller cost receive larger probability. As $\lambda \to 0$, $q_{\lambda}(x)$ converges to $q^{*}(x)$ for every $x$, so $q_{\lambda}$ is an approximation of $q^{*}$.

The normalizer $Z_{\lambda}$ adds one term for every binary vector, which is $2^{n}$ terms. Computing that sum is as hard as enumerating the whole search space, so $q_{\lambda}$ is not a practical sampler either.

### The trainable policy $p_{\theta}$

MCPG replaces $q_{\lambda}$ by a parameterized distribution $p_{\theta}$. The parameter $\theta$ is what training adjusts. The policy is built so that samples can be drawn without computing $Z_{\lambda}$, and the training objective pushes $p_{\theta}$ toward $q_{\lambda}$. Those samples are the candidate solutions of $\min f(x)$.

## $q^{*}$ and $q_{\lambda}$
We have noticed that $q^{*}$ is the ideal distribution where only the optimal $x\in X^{*}$ have the uniform probability on $q^{*}$. An example for $q^{*}$ is below.

<img src="fig_q_star.jpg" alt="The ideal distribution q-star on eight spin states" width="100%">

This example has $x \in \{-1,+1\}^3$ and two optimal states, $(-1,+1,+1)$ and $(+1,-1,+1)$. Since $X^{*}$ has only two states, each takes probability $\frac{1}{2}$. If $X^{*}$ had three states, each would take $\frac{1}{3}$. 

$q_{\lambda}$ which is:
$$
q_{\lambda}(x) = \frac{1}{Z_{\lambda}} \exp\!\left(-\frac{f(x)}{\lambda}\right),
\qquad
Z_{\lambda} = \sum_{x \in \{-1,+1\}^{n}} \exp\!\left(-\frac{f(x)}{\lambda}\right).
$$

How $q^{*}$ = $q_{\lambda}$ as $\lambda \to 0$:

$$
q_{\lambda}(x) = \frac{\exp\!\left(-\frac{f(x)}{\lambda}\right)}{\sum_{x \in \{-1,+1\}^{n}} \exp\!\left(-\frac{f(x)}{\lambda}\right)}
\qquad
$$

Let $f^{*}$ be the optimal min value of $f(x)$, $B_{n} = \{-1,+1\}^{n}$

multiply $\frac{\exp{\frac{f^{*}}{\lambda}}}{exp{\frac{f^{*}}{\lambda}}}$ to $q_{\lambda}(x)$

$$
q_{\lambda}(x) = \frac{\exp\!\left(\frac{f^{*} - f(x)}{\lambda}\right)}{\sum_{x \in B_n} \exp\!\left(\frac{f^{*} - f(x)}{\lambda}\right)}
\qquad
$$

For $x \in X^*$, $f(x) = f^*$ thus $\exp\!\left(\frac{f^{*} - f(x)}{\lambda}\right)$ = $\exp0 = 1$. So we can pull out these $x\in X^*$.

$$
q_{\lambda}(x) = \frac{\exp\!\left(\frac{f^{*} - f(x)}{\lambda}\right)}{|X^*| +  \sum_{x \in B_n \setminus X^*} \exp\!\left(\frac{f^{*} - f(x)}{\lambda}\right)}
\qquad
$$

Let $\lambda \to 0$, lets look at $\exp\!\left(\frac{f^{*} - f(x)}{\lambda}\right)$. 
Noticed that $f^*$ is this min value of $f(x)$ thus $f^* \leq f(x) \to f^* - f(x) \leq 0$
For cases $x \in B_n \setminus X^*$, $f^* - f(x) <0$ thus $\exp\!\left(\frac{-\epsilon}{\lambda}\right)$ where $\epsilon > 0$ since $\lambda \to 0^+$ so $\frac{-\epsilon}{\lambda} \to -\infty$ therefore $\exp\!\left(\frac{f^{*} - f(x)}{\lambda}\right) \to 0$

Thus for $x \in B_n \setminus X^*$, $q_{\lambda \to 0^+}(x) = \frac{0}{|X^*| + 0} = 0$
For cases $x \in X^*$, $q_{\lambda \to 0^+}(x) = \frac{1}{|X^*| + 0} = \frac{1}{|X^*|}$

So we can conclude that as $\lambda \to 0^+, q^* = q_{\lambda}$.



## $p_{\theta}(x)$

Why we need $p_{\theta}(x)$ instead of $q_{\lambda}$?  
Look at the term $Z_{\lambda}$ in $q_{\lambda}$: 
$$Z_{\lambda} = \sum_{x \in \{-1,+1\}^{n}} \exp\!\left(-\frac{f(x)}{\lambda}\right)$$
To compute this term requires look at every possible $x$ which there are $2^n$ possible $x$. This means the cost for computing $Z_{\lambda}$ is $O(2^n)$ since it increased exponentially is hard to sample. 

So we need to train $\theta$ so that $p_{\theta}$ approximately $q_{\lambda}$.

$p_{\theta}$ is defined as:
$$
p_{\theta}(x) = \prod_{i=1}^{n} \mu_i^{\frac{1+x_i}{2}} \, (1-\mu_i)^{\frac{1-x_i}{2}},
\qquad
\mu_i = \frac{1}{1 + e^{-\theta_i}}
$$

Let understand this equation step by step:

$x$ = $\{x_1, x_2,...,x_n\}$

$u_i$ = $p(x_i = +1)$ ==> when $x_i = 1$ $\frac{1+x_i}{2} = 1$, $\frac{1-x_i}{2} = 0$ so it keep $u_i$

$1-u_i$ = $p(x_i = -1)$ ==> when $x_i = -1$ $\frac{1+x_i}{2} = 0$, $\frac{1-x_i}{2} = 1$ so it keep $1 -u_i$

For example $u_i = 0.3$, $x = \{+1,-1,+1,-1\}$

$p_{\theta}(x) = p(x_1 = +1)p(x_2 = -1)p(x_3 = +1)p(x_4 = -1) ==> 0.3 * 0.7 * 0.3 * 0.7$

Each factor $\mu_i^{\frac{1+x_i}{2}} (1-\mu_i)^{\frac{1-x_i}{2}}$ is a Bernoulli distribution for one spin.
Writing $p_{\theta}$ as a product means we assume the spins $x_i$ are independent of each other.
This assumption is called the **mean field (MF) approximation**.

What if we do not use MF? Any distribution can be written with the chain rule:

$$
p(x) = p(x_1)\, p(x_2 \mid x_1)\, p(x_3 \mid x_1, x_2) \cdots p(x_n \mid x_1, \dots, x_{n-1})
$$

The factor $p(x_k \mid x_1, \dots, x_{k-1})$ needs one probability for each of the $2^{k-1}$ possible values of the earlier spins,
so a general distribution needs $2^n - 1$ numbers in total. With MF we only need $n$ numbers, $\theta_1, \dots, \theta_n$.

## KL divergence

Now we have know what is $q_{\lambda}$ and $p_{\theta}$ and our goal is to train $\theta$ so that $p_{\theta}$ approximately $q_{\lambda}$.

To do that we have to measure the distance between $q_{\lambda}$ and $p_{\theta}$ and minimizing the distance. 

The KL divergence is used to measure the distance between $q_{\lambda}$ and $p_{\theta}$ and is defined as:
$$
\mathrm{KL}(p_{\theta} \parallel q_{\lambda})
= \sum_{x \in B_n} p_{\theta}(x) \log \frac{p_{\theta}(x)}{q_{\lambda}(x)}.
$$

Recall $q_{\lambda}(x) = \frac{1}{Z_{\lambda}} \exp\!\left(-\frac{f(x)}{\lambda}\right)$, we are keep using the the property $\log \frac{a}{b} = \log a - \log b$ and $\log ab = \log a + \log b$ below

$$
\mathrm{KL}(p_{\theta} \parallel q_{\lambda})
= \sum_{x \in B_n} p_{\theta}(x) \log \frac{p_{\theta}(x)}{\frac{1}{Z_{\lambda}} \exp\!\left(-\frac{f(x)}{\lambda}\right)}.
$$

$$
\mathrm{KL}(p_{\theta} \parallel q_{\lambda})
= \sum_{x \in B_n} p_{\theta}(x) \log p_{\theta}(x) - \sum_{x \in B_n} p_{\theta}(x) \log \frac{1}{Z_{\lambda}} \exp\!\left(-\frac{f(x)}{\lambda}\right).
$$

$$
\mathrm{KL}(p_{\theta} \parallel q_{\lambda})
= \sum_{x \in B_n} p_{\theta}(x) \log p_{\theta}(x) - \sum_{x \in B_n} p_{\theta}(x) \log \frac{1}{Z_{\lambda}}- \sum_{x \in B_n} p_{\theta}(x) \log  \exp\!\left(-\frac{f(x)}{\lambda}\right).
$$

Notice $\ln e^{a} = a$ so $\log \exp\!\left(-\frac{f(x)}{\lambda}\right)$ = $-\frac{f(x)}{\lambda}$

$$
\mathrm{KL}(p_{\theta} \parallel q_{\lambda})
= \sum_{x \in B_n} p_{\theta}(x) \log p_{\theta}(x) + \sum_{x \in B_n} p_{\theta}(x) \log Z_{\lambda} + \sum_{x \in B_n} p_{\theta}(x) \frac{f(x)}{\lambda}.
$$

$\sum_{x \in B_n} p_{\theta}(x) = 1$ thus we have end it up in the term:
$$\mathrm{KL}(p_{\theta} \parallel q_{\lambda})
= \frac{1}{\lambda} \sum_{x \in B_n} p_{\theta}(x) f(x)
+ \sum_{x \in B_n} p_{\theta}(x) \log p_{\theta}(x)
+ \log Z_{\lambda}.
$$

Our goal is to find the $\theta$ so that min the distance

$$
\min_{\theta} \mathrm{KL}(p_{\theta} \parallel q_{\lambda})
= \frac{1}{\lambda} \sum_{x \in B_n} p_{\theta}(x) f(x)
+ \sum_{x \in B_n} p_{\theta}(x) \log p_{\theta}(x)
+ \log Z_{\lambda}.
$$

$$
= \frac{1}{\lambda}\left(\mathbb{E}_{p_{\theta}}[f(x)] + \lambda\,\mathbb{E}_{p_{\theta}}[\log p_{\theta}(x)]\right) + \log Z_{\lambda}.
$$

Notice $\log Z_{\lambda}$ is a constant so it is  equivalent to min the term 
$$
\min_{\theta} L_{\lambda}(\theta) =\mathbb{E}_{p_{\theta}}[f(x)] + \lambda\,\mathbb{E}_{p_{\theta}}[\log p_{\theta}(x)].
$$

## Summary of set up:

Ideal distribution 

$$
q^{*}(x) =
\begin{cases}
1/|X^{*}| & \text{if } x \in X^{*}, \\
0 & \text{if } x \notin X^{*}.
\end{cases}
$$

The true distribution(Gibbs distribution) $q_{\lambda}(x)$

$$
q_{\lambda}(x) = \frac{1}{Z_{\lambda}} \exp\!\left(-\frac{f(x)}{\lambda}\right),
\qquad
Z_{\lambda} = \sum_{x \in \{-1,+1\}^{n}} \exp\!\left(-\frac{f(x)}{\lambda}\right).
$$

The trainable distribution $p_{\theta}$

$$
p_{\theta}(x) = \prod_{i=1}^{n} \mu_i^{\frac{1+x_i}{2}} \, (1-\mu_i)^{\frac{1-x_i}{2}},
\qquad
\mu_i = \frac{1}{1 + e^{-\theta_i}}
$$

KL divergence use to measure the distance between $p_{\theta}$ and $q_{\lambda}$

$$
\mathrm{KL}(p_{\theta} \parallel q_{\lambda})
= \sum_{x \in B_n} p_{\theta}(x) \log \frac{p_{\theta}(x)}{q_{\lambda}(x)}.
$$



## Part 2 Algorithm

<img src="fig1_mcpg_pipeline.png" alt="Fig. 1 The pipeline of MCPG" width="100%">

**Fig. 1** The pipeline of MCPG. Starting points are drawn by MCMC from the mean-field policy $p_\theta(\cdot \mid \mathcal{P})$. A filter $T$ turns those raw samples into locally improved samples, the best of which become the next starting points. The same filtered samples supply the advantage $A_i^j$ and the policy gradient $\bar g$, which updates $\theta$.

**Figure source:** Chen, C., Chen, R., Li, T., Ao, R., & Wen, Z. (2023). *A Monte Carlo Policy Gradient Method with Local Search for Binary Optimization*. [arXiv:2307.00783](https://arxiv.org/abs/2307.00783), Figure 1.

Let's step by step walk through this images:

1. Starting by given a problem Instance $\mathbf{P}$. This $\mathbf{P}$ it is just saying given a f(x) find x such that $\min_{x \in B_n} f(x)$

2. Based on the Problem Instance we use Mean Field Policy to generate a Distribution, the starting $\theta_i = 0$
$$
p_{\theta}(x) = \prod_{i=1}^{n} \mu_i^{\frac{1+x_i}{2}} \, (1-\mu_i)^{\frac{1-x_i}{2}},
\qquad
\mu_i = \frac{1}{1 + e^{-\theta_i}}
$$

3. Randomly generate K starting points which is $S_1 ... S_k$

4. Use MCMC for these starting points it will generate a bach for each $s_j$ which becomes $\{s_1^1,\ldots,s_1^m\}\ \cdots\ \{s_k^1,\ldots,s_k^m\}$ is call the raw samples

5. From the raw samples apply the filter function to get $\{\hat{s}_1^1,\ldots,\hat{s}_1^m\}\ \cdots\ \{\hat{s}_k^1,\ldots,\hat{s}_k^m\}$

6. Use both the raw samples and the filted samples to get the gradiant $\bar{g}$ 

7. Use $\bar{g}$ to update the $\theta$ which is $\theta \leftarrow \theta - \eta\bar{g}$

8. Pick the best $\hat{s}$ in each bach use it as a new starting points for the next iteration.

9. Go back to step 4

Finally, after $N$ iterations, this algorithm will return the best solution found across all iterations.

## The gradient

Recall from KL divergence we get:
$$
\min_{\theta} L_{\lambda}(\theta) =\mathbb{E}_{p_{\theta}}[f(x)] + \lambda\,\mathbb{E}_{p_{\theta}}[\log p_{\theta}(x)].
$$

BY taking the gradint respect to $\theta$ we have:
$$
\nabla_{\theta} L_{\lambda}(\theta)
= \nabla_{\theta} \sum_{x \in B_n} p_{\theta}(x)\left(f(x) + \lambda \log p_{\theta}(x)\right).
$$

BY the product rule $(xy)' = x'y + xy'$ we have:

$$
\nabla_{\theta} L_{\lambda}(\theta) = \lambda \sum_{x \in B_n} p_{\theta}(x) \nabla_{\theta} \log p_{\theta}(x)
+ \sum_{x \in B_n} \left(f(x) + \lambda \log p_{\theta}(x)\right) \nabla_{\theta} p_{\theta}(x).
$$

Noticed:
$$
\nabla_{\theta}\log p_{\theta}(x) = \frac{1}{p_{\theta}(x) } \nabla_{\theta} p_{\theta}(x) 
$$

$$
p_{\theta}(x) \nabla_{\theta} \log p_{\theta}(x) = \nabla_{\theta} p_{\theta}(x)
$$

So we get:

$$
\lambda \sum_{x \in B_n} \nabla_{\theta} p_{\theta}(x)
+ \sum_{x \in B_n} \left(f(x) + \lambda \log p_{\theta}(x)\right) p_{\theta}(x) \nabla_{\theta} \log p_{\theta}(x).
$$

Since
$$
\lambda \sum_{x \in B_n} \nabla_{\theta} p_{\theta}(x) =  \lambda \nabla_{\theta} \sum_{x \in B_n} p_{\theta}(x)1 =  \lambda \nabla_{\theta} 1 = 0 
$$

So left by:

$$
\sum_{x \in B_n} \left(f(x) + \lambda \log p_{\theta}(x)\right) p_{\theta}(x) \nabla_{\theta} \log p_{\theta}(x).
$$

Which is:
$$
\mathbb{E}_{p_{\theta}}\left[\left(f(x) + \lambda \log p_{\theta}(x)\right) \nabla_{\theta} \log p_{\theta}(x)\right].
$$


In the paper, it also add a c in to the gradiant which is:

$$
\mathbb{E}_{p_{\theta}}\left[\left(f(x) + \lambda \log p_{\theta}(x) + c\right) \nabla_{\theta} \log p_{\theta}(x)\right]
=
\mathbb{E}_{p_{\theta}}\left[\left(f(x) + \lambda \log p_{\theta}(x)\right) \nabla_{\theta} \log p_{\theta}(x)\right] + \mathbb{E}_{p_{\theta}}[c\nabla_{\theta} \log p_{\theta}(x)]
$$

Where
$$
\mathbb{E}_{p_{\theta}}[c\nabla_{\theta} \log p_{\theta}(x)] 
= c\sum_{x \in B_n} \nabla_{\theta} p_{\theta}(x)
= c\nabla_{\theta} \sum_{x \in B_n} p_{\theta}(x)
= c\nabla_{\theta} 1
= 0.
$$

And $c = \mathbb{E}_{p_{\theta}}[f(x)].$ 


So the gradint for the loss function is

$$
\mathbb{E}_{p_{\theta}}\left[\left(f(x) + \lambda \log p_{\theta}(x) + c\right) \nabla_{\theta} \log p_{\theta}(x)\right]
$$
We will talk about how to compute this later


## Filter Function
The goal for the filter function is to find the optimal $\hat{x}$ in the neighborhood($N(x)$) of $x$ 

$$
T(x) \in \arg\min_{\hat{x} \in N(x)} f(\hat{x}).
$$

So we need to defined the set $N(x)$ which $N(x) \subset B_n$ and $|N(x)| \geq 2$

There are two methods provided in the paper for the filter function but we will fucsed on the method call local search in this tutorial:

<img src="fig_local_search.png" alt="Algorithm 2 Local Search" width="100%">

**Figure source:** Chen, C., Chen, R., Li, T., Ao, R., & Wen, Z. (2023). *A Monte Carlo Policy Gradient Method with Local Search for Binary Optimization*. [arXiv:2307.00783](https://arxiv.org/abs/2307.00783), Algorithm 2.


Let's step by step go through what is this algorithm doing. 
 
1. For a specifc $x = {x_1, x_2, ... x_n}$ this $x$ has n bit and each bit are in $\{+1, -1\}$

2. Starting at the first bit $x_i, i = 1$

3. For bit $x_i$ you filp the bit which if $x_i = -1$ you flip to $x_i = +1$ 

4. Then you can check if this action of filping the bit improve or not by pulg the filped $x$ into $f(x)$

5. If improved we will keep the $x$ with this action, if not we will keep the x before this action.

6. move to the next bit $i = i+1$ 

7. go back to step 3

8. Once you finished all the filpping for n bits you end up with a $x$ that is the min in $N(x)$



## Note that I take when I am reading the paper:

2.3 Prototype Algorithm
$$
\mathbb{E}_{p_{\theta}}[\nabla_{\theta} \log p_{\theta}(x)]
= \sum_{x \in B_n} \nabla_{\theta} p_{\theta}(x)
= \nabla_{\theta} \sum_{x \in B_n} p_{\theta}(x)
= \nabla_{\theta} 1
= 0.
$$

$$
\mathbb{E}_{p_{\theta}}[\nabla_{\theta} \log p_{\theta}(x)] = \sum_{x \in B_n} p_{\theta}(x) \nabla_{\theta}\log p_{\theta}(x)
$$

$$
\nabla_{\theta}\log p_{\theta}(x) = \frac{1}{p_{\theta}(x) } \nabla_{\theta} p_{\theta}(x) 
$$

$$
\mathbb{E}_{p_{\theta}}[\nabla_{\theta} \log p_{\theta}(x)] = \sum_{x \in B_n}  \nabla_{\theta}p_{\theta}(x)
$$